# Exp 1 - Simulation of Hard, Soft, and Firm Real-Time Tasks using Python

> Scope note: this notebook is a lab-scale deterministic simulation. It is intended for understanding timing, communication, and security concepts. It is not a certification model for a production autonomous vehicle.

## Objective

Model three real-time task classes and evaluate how missed deadlines affect system behaviour.

- **Hard real-time task:** a missed deadline is treated as failure because the system may become unsafe.
- **Firm real-time task:** a late result is discarded because it no longer represents the current state.
- **Soft real-time task:** a late result is still usable, but its quality or value is reduced.

In an autonomous-system context, this distinction matters because the same timing miss can have different consequences depending on whether the task controls braking, planning, logging, display, or telemetry.

## Textbook Notes and Case Studies

### 1. Textbook Background

Real-time computing is not only about producing the correct answer; it is about producing the correct answer before a specified deadline. In an autonomous system, a late result can be useless or unsafe even when the calculation itself is mathematically correct. A braking decision, lane-change warning, collision alert, motor-control signal, or network retransmission decision must be evaluated by both logical correctness and timing correctness.

Real-time tasks are usually classified into three categories:

| Task Type | Deadline Meaning | Result After Deadline | Example |
|---|---:|---|---|
| Hard real-time | Must never be missed | Failure condition | Brake actuation, airbag trigger, flight-control loop |
| Firm real-time | Occasional miss may be tolerated | Late result has zero value | Object detection frame after vehicle has passed the obstacle |
| Soft real-time | Miss degrades quality | Late result still has reduced value | Infotainment update, dashboard refresh, non-critical telemetry |

The main difference is the value function of the output. A hard real-time task has a discontinuous safety boundary: after the deadline, the system may be considered failed. A firm real-time task has an economic or usefulness boundary: the result is discarded. A soft real-time task has graceful degradation: the result may still improve comfort or monitoring.

### 2. Architecture Notes

```
Sensor/Event Source
      |
      v
Task Release Queue -> Scheduler -> CPU/Processor -> Completion Log
      |                  |              |
      |                  v              v
      |            Priority/Policy   Deadline Check
      |                                 |
      v                                 v
Deadline Table                  Miss Ratio / Utility
```

The release queue contains tasks waiting to execute. The scheduler decides the execution order. The processor executes the selected task. The completion log records finish time, deadline status, and whether the output is accepted or discarded. This architecture is simple, but it captures the central problem in embedded real-time design: limited processing capacity must be assigned to tasks with different timing criticality.

### 3. Important Formulas

If a task is released at time r and has relative deadline D, its absolute deadline is:

```
absolute_deadline = r + D
```

A task meets its deadline when:

```
finish_time <= absolute_deadline
```

Deadline miss ratio:

```
miss_ratio = missed_tasks / total_tasks
```

Satisfaction ratio:

```
satisfaction_ratio = completed_before_deadline / total_tasks
```

For a firm real-time task, useful work can be estimated as:

```
useful_work_ratio = useful_completed_tasks / total_completed_tasks
```

This matters because a processor may appear busy, but if many firm tasks complete late, the useful throughput is lower than the raw completed-task count.

### 4. Classroom Case Studies

Case Study A - Autonomous Emergency Braking:
A front camera detects an obstacle and the controller must decide whether to brake. This is hard real-time because the command loses safety meaning if it arrives after the stopping window. The correct analysis is not only whether the obstacle was detected, but whether the perception, decision, and actuation chain completed before the deadline.

Case Study B - Traffic-Sign Recognition:
If a sign-recognition output arrives late after the vehicle already passed the sign, the result may be discarded. This is a firm real-time example. The algorithm may be accurate, but timeliness decides whether the result is useful.

Case Study C - Cabin Display Update:
A speedometer animation or non-critical dashboard visualization can tolerate occasional delay. The user may notice reduced smoothness, but the system is not automatically unsafe. This is soft real-time behavior.

### 5. Viva and Exam Notes

Do not write that real-time means fast. A slow system can be real-time if it always meets its deadline, while a fast system can fail if it occasionally misses a critical deadline. In lab analysis, always report release time, execution time, deadline, finish time, deadline status, and miss ratio.

### 6. Source Notes

- Python timing functions used for experiments are documented in the official Python `time` module documentation: https://docs.python.org/3/library/time.html
- The classic utilization bound for fixed-priority periodic scheduling comes from Liu and Layland, 1973: https://dl.acm.org/doi/10.1145/321738.321743


## Detailed Notes

### 1. Real-Time Task Classes

Real-time systems are judged not only by whether a computation is correct, but also by whether the answer is produced before its deadline.

| Task Class | Meaning | Deadline Miss Effect | Autonomous-System Example |
|---|---|---|---|
| Hard real-time | Deadline must always be met | Unsafe or failed system behaviour | Emergency braking, steering actuation |
| Firm real-time | Late output has no value | Result is discarded | Stale obstacle-detection frame |
| Soft real-time | Late output still has partial value | Quality degrades | Telemetry display, video stream |

### 2. Architecture Explanation

The experiment models a small real-time execution pipeline:

```text
Task Definition
  -> Scheduler
  -> Execution Timeline
  -> Deadline Checker
  -> Class-wise Result Analysis
```

Each task contains a release time, execution time, relative deadline, and task class. The scheduler simulates when the task starts and finishes. The deadline checker decides whether the output is on time. The final analysis explains the result differently for hard, firm, and soft tasks.

### 3. Key Timing Terms

- **Release time (R):** time at which the task becomes ready.
- **Execution time (C):** processor time required by the task.
- **Relative deadline (Drel):** allowed time after release.
- **Absolute deadline (Dabs):** final completion time allowed on the global timeline.
- **Finish time (F):** time at which the task actually completes.

### 4. Formulas

Absolute deadline:

\[
D_{abs} = R + D_{rel}
\]

Deadline condition:

\[
F \le D_{abs}
\]

Deadline miss ratio:

\[
\text{miss ratio} = \frac{\text{number of missed tasks}}{\text{total number of tasks}}
\]

Deadline satisfaction ratio:

\[
\text{satisfaction ratio} = \frac{\text{number of tasks completed before deadline}}{\text{total number of tasks}}
\]

Useful-work score used in the post-lab:

\[
\text{useful work ratio} = \frac{\sum \text{useful credit}}{\text{number of tasks}}
\]

The notebook gives full credit to on-time results, no credit to missed hard/firm tasks, and partial credit to missed soft tasks.

### 5. In-Lab Interpretation

The in-lab simulation uses a simple non-preemptive scheduler. This means tasks execute one after another. If early jobs consume too much time, later jobs can miss their deadlines.

Important observation:

- Hard tasks should be designed so their deadlines are met even under load.
- Firm tasks can complete late, but their results are useless after the deadline.
- Soft tasks can complete late and still provide partial value.

### 6. Post-Lab Interpretation

The post-lab experiment increases system load from 40% to 120%. Increasing load increases actual execution time. When the system becomes overloaded, the miss ratio rises.

What to observe:

- At low load, all task classes usually meet deadlines.
- Near full load, deadline misses begin appearing.
- In overload, firm and soft tasks lose useful output.
- A hard-task miss is unacceptable even if the numerical miss ratio is small.

### 7. Practical Design Notes

For an autonomous system, the scheduler should not treat all tasks equally. Safety-critical tasks need stronger guarantees through priority assignment, admission control, execution-time budgeting, or dedicated compute resources. Non-critical telemetry or display tasks can be degraded first during overload.

### 8. Precautions

- Do not interpret this simple scheduler as a production RTOS scheduler.
- Use worst-case execution time, not average execution time, for safety-critical design.
- Validate deadline assumptions with real measurements before using them in a deployed system.
- Keep hard real-time workloads small and predictable.


## Architecture

```text
Task Set
  |-- task name
  |-- class: hard / firm / soft
  |-- release time
  |-- execution time
  |-- relative deadline
          |
          v
Simple Scheduler
  |-- waits until task release
  |-- executes task for its execution time
  |-- records finish time
          |
          v
Deadline Evaluator
  |-- absolute deadline = release time + relative deadline
  |-- deadline met if finish time <= absolute deadline
          |
          v
Result Table + Class-wise Miss Ratio
```

The notebook uses a non-preemptive scheduler so the timing effect is easy to inspect. In a real system, higher-priority hard real-time tasks would normally receive stricter scheduling guarantees.

## Formulas and Required Theory

For a task \(i\):

\[
D_i^{abs} = R_i + D_i^{rel}
\]

\[
\text{deadline met}_i =
\begin{cases}
1, & F_i \le D_i^{abs}\\
0, & F_i > D_i^{abs}
\end{cases}
\]

\[
\text{deadline miss ratio} = \frac{\text{number of missed jobs}}{\text{total number of jobs}}
\]

\[
\text{useful work ratio} =
\frac{\sum \text{useful credit per completed job}}{\text{total jobs}}
\]

Useful-credit rule used in this notebook:

- on-time result: `1.00`
- late hard result: `0.00`
- late firm result: `0.00`
- late soft result: `0.55`, representing degraded but still partially useful output

## In-Lab Method

1. Define task parameters for representative autonomous-system workloads.
2. Execute the task list in scheduler order.
3. Compute finish time for each task.
4. Compare finish time against absolute deadline.
5. Report per-task status and class-wise miss ratio.

Expected observation: hard tasks should be protected from misses. Firm and soft tasks may miss under overload, but the interpretation differs: firm results are discarded while soft results degrade.

In [1]:
tasks = [
    {"name": "Brake actuator command", "class": "Hard", "release": 0, "exec": 6, "deadline": 8},
    {"name": "Collision-warning fusion", "class": "Hard", "release": 2, "exec": 5, "deadline": 12},
    {"name": "Trajectory replanning", "class": "Firm", "release": 4, "exec": 9, "deadline": 14},
    {"name": "HD map refresh", "class": "Firm", "release": 7, "exec": 12, "deadline": 18},
    {"name": "Cabin telemetry update", "class": "Soft", "release": 8, "exec": 15, "deadline": 20},
    {"name": "Video status stream", "class": "Soft", "release": 10, "exec": 18, "deadline": 22},
]

clock = 0
results = []
for task in tasks:
    start = max(clock, task["release"])
    finish = start + task["exec"]
    absolute_deadline = task["release"] + task["deadline"]
    met = finish <= absolute_deadline
    results.append((task["name"], task["class"], task["release"], task["exec"], absolute_deadline, finish, "Met" if met else "Missed"))
    clock = finish

print("EXP 1 - IN-LAB RESULT")
print(f"{'Task':32} {'Class':6} {'Rel':>4} {'Exec':>5} {'Dead':>5} {'Fin':>5} {'Status':>8}")
for row in results:
    print(f"{row[0][:32]:32} {row[1]:6} {row[2]:4} {row[3]:5} {row[4]:5} {row[5]:5} {row[6]:>8}")
for cls in ["Hard", "Firm", "Soft"]:
    subset = [r for r in results if r[1] == cls]
    misses = sum(r[-1] == "Missed" for r in subset)
    print(f"{cls} miss ratio = {misses / len(subset):.2f}")

EXP 1 - IN-LAB RESULT
Task                             Class   Rel  Exec  Dead   Fin   Status
Brake actuator command           Hard      0     6     8     6      Met
Collision-warning fusion         Hard      2     5    14    11      Met
Trajectory replanning            Firm      4     9    18    20   Missed
HD map refresh                   Firm      7    12    25    32   Missed
Cabin telemetry update           Soft      8    15    28    47   Missed
Video status stream              Soft     10    18    32    65   Missed
Hard miss ratio = 0.00
Firm miss ratio = 1.00
Soft miss ratio = 1.00


## Post-Lab Method

The post-lab cell generates 100 mixed tasks at several load levels. Load scales actual execution time. This shows how increasing computational demand affects deadline misses and useful work.

Interpretation rule:

- A rising miss ratio means the system is approaching overload.
- A hard-task miss ratio above zero is unacceptable in a safety-critical design.
- A firm-task miss reduces useful output sharply.
- A soft-task miss reduces quality gradually.

In [2]:
import random

def run_trial(load, seed=341401):
    rng = random.Random(seed)
    factors = {"Hard": 1.75, "Firm": 1.35, "Soft": 1.25}
    counts = {k: 0 for k in factors}
    misses = {k: 0 for k in factors}
    useful = {k: 0.0 for k in factors}
    for i in range(100):
        cls = ["Hard", "Firm", "Soft"][i % 3]
        base = rng.uniform(4, 18)
        actual = base * load * rng.uniform(0.90, 1.35)
        met = actual <= base * factors[cls]
        counts[cls] += 1
        misses[cls] += 0 if met else 1
        useful[cls] += 1.0 if met else (0.55 if cls == "Soft" else 0.0)
    return {cls: (misses[cls] / counts[cls], useful[cls] / counts[cls]) for cls in counts}

print("EXP 1 - POST-LAB LOAD SWEEP")
print(f"{'Load':>6} {'Class':>6} {'Miss ratio':>12} {'Useful work':>12} {'Plot':>10}")
for load in [0.40, 0.60, 0.80, 1.00, 1.20]:
    for cls, values in run_trial(load).items():
        miss, useful = values
        plot = "-" if miss == 0 else "#" * max(1, round(miss * 10))
        print(f"{load:6.0%} {cls:>6} {miss:12.2f} {useful:12.2f} {plot:>10}")

EXP 1 - POST-LAB LOAD SWEEP
  Load  Class   Miss ratio  Useful work       Plot
   40%   Hard         0.00         1.00          -
   40%   Firm         0.00         1.00          -
   40%   Soft         0.00         1.00          -
   60%   Hard         0.00         1.00          -
   60%   Firm         0.00         1.00          -
   60%   Soft         0.00         1.00          -
   80%   Hard         0.00         1.00          -
   80%   Firm         0.00         1.00          -
   80%   Soft         0.00         1.00          -
  100%   Hard         0.00         1.00          -
  100%   Firm         0.00         1.00          -
  100%   Soft         0.09         0.96          #
  120%   Hard         0.00         1.00          -
  120%   Firm         0.61         0.39     ######
  120%   Soft         0.61         0.73     ######


## What to Write in the Lab Record

- Copy the in-lab result table.
- Record which classes met or missed deadlines.
- Explain why hard tasks need the strongest timing guarantee.
- In the post-lab section, compare miss ratio and useful work at 40%, 60%, 80%, 100%, and 120% load.
- Conclude that deadline classification changes the engineering response to overload.

## References

- Python `time` module documentation: https://docs.python.org/3/library/time.html
- Python `statistics` module documentation: https://docs.python.org/3/library/statistics.html
- Python `random` module documentation: https://docs.python.org/3/library/random.html